# LiveSense Final Validation Notebook

This notebook documents and exercises the final LiveSense safety rules without opening the webcam. It focuses on fresh-frame sleep timing, blink rejection, yaw suppression during eating/drinking, and positioned object cues.

In [ ]:
from dataclasses import asdict

from camera import classify_distracted_activity
from config import load_settings
from signals import DrowsinessMonitor
from vision import classify_object_cues

settings = load_settings()
settings

## Two-second sleep progression

Only fresh face-landmark results advance this timer in the live processor. With continuously closed eyes, the expected progression is Awake, Dozing, then Sleeping at two seconds.

In [ ]:
sleep_monitor = DrowsinessMonitor(
    dozing_seconds=settings.monitoring.dozing_seconds,
    sleeping_seconds=settings.monitoring.sleeping_seconds,
)

sleep_progression = []
for timestamp in (0.0, 1.0, 2.0):
    result = sleep_monitor.update(
        timestamp=timestamp,
        face_detected=True,
        eyes_closed=True,
        yawning=False,
        head_pitch=0.0,
    )
    sleep_progression.append(asdict(result))

sleep_progression

## Blink rejection

A short closed-eye observation followed by open eyes must not activate the alarm or accumulate toward a later sleep event.

In [ ]:
blink_monitor = DrowsinessMonitor()
blink_monitor.update(
    timestamp=0.0, face_detected=True, eyes_closed=True,
    yawning=False, head_pitch=0.0,
)
blink_result = blink_monitor.update(
    timestamp=0.4, face_detected=True, eyes_closed=False,
    yawning=False, head_pitch=0.0,
)

assert blink_result.alarm_active is False
assert blink_result.state == "Awake"
asdict(blink_result)

## Eating and drinking evidence

Eating may use repeated food-object evidence or a sustained hand-to-mouth plus jaw-opening cue. Drinking requires repeated cup/bottle evidence near the mouth. These cues are independent and can suppress a conflicting mouth-open yawn interpretation.

In [ ]:
activity_examples = {
    "neutral": classify_distracted_activity(
        hand_near_ear=False, hand_near_mouth=False, mouth_open_score=0.1,
        phone_object_score=0, drink_object_score=0, food_object_score=0,
    ),
    "eating_gesture": classify_distracted_activity(
        hand_near_ear=False, hand_near_mouth=True, mouth_open_score=0.35,
        phone_object_score=0, drink_object_score=0, food_object_score=0,
    ),
    "repeated_food": classify_distracted_activity(
        hand_near_ear=False, hand_near_mouth=False, mouth_open_score=0.1,
        phone_object_score=0, drink_object_score=0, food_object_score=2,
    ),
    "repeated_drink": classify_distracted_activity(
        hand_near_ear=False, hand_near_mouth=True, mouth_open_score=0.1,
        phone_object_score=0, drink_object_score=2, food_object_score=0,
    ),
}

activity_examples  # tuple order: phone evidence, drinking, eating

## Object position filtering

Object labels alone are insufficient. LiveSense also checks whether the detected food, cup, or bottle is close to the estimated mouth.

In [ ]:
face_box = (100, 40, 80, 100)
near_mouth = classify_object_cues(
    [("cup", 0.70, (128, 110, 24, 32))],
    face_box,
)
far_from_mouth = classify_object_cues(
    [("cup", 0.70, (10, 180, 24, 32))],
    face_box,
)

{
    "near_mouth": asdict(near_mouth),
    "far_from_mouth": asdict(far_from_mouth),
}

## Run the final dashboard

From PowerShell in the project directory:

```powershell
.venv\Scripts\Activate.ps1
uvicorn app:app --host 127.0.0.1 --port 8501
```

Open `http://127.0.0.1:8501`, start the camera, and keep the face, shoulders, hands, and relevant objects clearly visible. A browser refresh starts a new in-memory session.

## Final validation commands

```powershell
pytest
ruff check .
python -m pip check
node --check web\app.js
node --check web\reports.js
```

LiveSense is a visual development system, not a certified automotive or medical safety device.